# Day 41: CUDA Profiling with torch.profiler

**Layer:** Implementation


## Concept Overview

torch.profiler captures GPU kernel execution timelines, memory allocation events, and CPU-GPU synchronization points. It's the primary tool for identifying bottlenecks in PyTorch models.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. CUDA Profiling with torch.profiler

torch.profiler captures CPU and CUDA events, showing kernel timing, memory allocation, and CPU-GPU sync points. The Chrome trace format enables visual inspection of the execution timeline.


In [ ]:
model = nn.Sequential(
    nn.Linear(1024, 4096), nn.GELU(),
    nn.Linear(4096, 1024), nn.GELU(),
    nn.Linear(1024, 512)
).to('cuda' if torch.cuda.is_available() else 'cpu').half()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
x = torch.randn(64, 1024, device=device, dtype=torch.float16)

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU,
                torch.profiler.ProfilerActivity.CUDA] if device=='cuda'
               else [torch.profiler.ProfilerActivity.CPU],
    record_shapes=True, with_flops=True,
    profile_memory=True,
) as prof:
    with torch.profiler.record_function('forward_pass'):
        for _ in range(20):
            y = model(x)

sort_key = 'cuda_time_total' if device=='cuda' else 'cpu_time_total'
print('Top kernels:')
print(prof.key_averages().table(sort_by=sort_key, row_limit=10))
prof.export_chrome_trace('/tmp/trace.json')
print('Trace saved to /tmp/trace.json (view at chrome://tracing)')


## 2. Interpreting Profiler Output

Key metrics: self_cuda_time (time in kernel, not counting called kernels), cuda_memory_usage, count (kernel launches). Identify bottlenecks by sorting on different metrics.


In [ ]:
# Simulate what the profiler output tells us
import json
print('How to read profiler output:')
metrics = [
    ('self_cuda_time', 'Time in the kernel itself (us)'),
    ('cuda_time_total', 'Total time including sub-kernels'),
    ('cpu_time_total', 'CPU overhead (dispatch, sync)'),
    ('cuda_memory_usage', 'Peak memory allocated in this op'),
    ('count', 'Number of times this kernel ran'),
    ('flops', 'FLOPs if computable'),
]
for metric, desc in metrics:
    print(f'  {metric:<25} {desc}')
print()
print('Bottleneck patterns to look for:')
patterns = [
    ('High CPU time, low CUDA time', 'Kernel launch overhead — fuse operations'),
    ('High cuda_memory_usage', 'Memory bottleneck — consider in-place ops'),
    ('Many short kernels', 'Too many small ops — use torch.compile'),
    ('Low FLOP efficiency', 'Memory-bound — increase batch size'),
]
for pattern, action in patterns:
    print(f'  {pattern:<40} → {action}')


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- torch.profiler captures GPU kernel execution timelines, memory allocation events, and CPU-GPU synchronization points.
- The Chrome trace export is the most powerful profiler feature: it shows kernel launch, execution, and memory allocation as a timeline, making CPU-GPU synchronization stalls immediately visible..
- Day 41 implementation complete.
